# Phase 4 — Diagnostic d'apprentissage du XGBoost (FD001)

Application de la **section 5** de [`docs/protocole_evaluation.md`](../docs/protocole_evaluation.md) au modèle XGBoost de la Phase 3
(46 features, `n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42`, plis `GroupKFold(n_splits=5, shuffle=True, random_state=42)`).

**Déjà fait en Phase 3 et non refait ici** (`notebooks/03_modelisation.ipynb` §12-13, `docs/03_modelisation.md` §6-7) :
écart RMSE train / validation (10.94 vs 16.86) et courbes de validation sur `max_depth` et `n_estimators`.
Ici, on retrouve seulement les 10.94 / 16.86 comme contrôle (sanity check), au point « 80 moteurs » de la courbe.

**Une seule question : plus de moteurs d'entraînement aideraient-ils ?**
On y répond par une **courbe d'apprentissage** : l'erreur selon le nombre de moteurs d'entraînement (10, 20, 40, 60, 80), validation toujours sur des moteurs tenus à part, plusieurs tirages par taille.
Elle sert à décider s'il vaut la peine d'ajouter FD003, et donne une procédure réutilisable pour un futur LSTM.

**Règles respectées** : le jeu de test officiel n'est jamais lu dans ce notebook ; aucun fichier `src/` modifié ; fonctions définies dans le notebook.

## 0. Pourquoi une courbe d'apprentissage ici (rappel court)

Les courbes de validation (Phase 3) font varier la **complexité du modèle** à données fixes. La courbe d'apprentissage fait varier la **quantité de données** à modèle fixe.
Seule elle peut répondre à « plus de moteurs aideraient-ils ? » (FD001 n'a que 100 moteurs ; FD003 en apporte 100 de plus).

| Ce qu'on observe | Lecture |
|---|---|
| l'erreur de validation baisse encore à 80 moteurs | plus de données aideraient |
| l'erreur de validation est déjà plate | la limite vient des features ou du modèle, pas du volume |
| l'écart train / validation reste grand quelle que soit la taille | sur-apprentissage persistant : régulariser |

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul,
)

FENETRE = 5         # fenetre glissante, identique au notebook 03
PLAFOND_RUL = 125   # plafond de la RUL, identique au notebook 03

# versions des bibliotheques, a noter dans le rapport (protocole section 1, point 6)
print(f"numpy {np.__version__}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}, xgboost {xgboost.__version__}")

## 1. Chargement et features (identiques au notebook 03)

On reproduit exactement le pipeline de `03_modelisation.ipynb` (cellules de chargement et de features) : mêmes 15 capteurs lissés, mêmes moyennes et écarts-types glissants sur 5 cycles, `time_cycles`, soit **46 features**.
Seul le fichier `train_FD001.txt` est chargé.

In [ ]:
def ajouter_features_glissantes(df, sensor_cols, fenetre):
    '''
    Ajoute, pour chaque capteur, deux colonnes calculees sur une fenetre glissante de cycles :
    la moyenne et l'ecart-type des `fenetre` derniers cycles, moteur par moteur.
    (Copie conforme de la fonction du notebook 03.)

    Parameters
    ----------
    df : pandas.DataFrame
        Le tableau source (une copie est renvoyee).
    sensor_cols : list de str
        Les capteurs pour lesquels calculer les features.
    fenetre : int
        Nombre de cycles de la fenetre glissante.

    Returns
    -------
    pandas.DataFrame
        Le tableau avec "<capteur>_glissante_moy" et "<capteur>_glissante_std" en plus.
    '''
    df = df.copy()  # copie independante

    for sensor_name in sensor_cols:  # un capteur a la fois
        colonne_moyenne = sensor_name + "_glissante_moy"
        colonne_ecart_type = sensor_name + "_glissante_std"
        df[colonne_moyenne] = 0.0  # valeurs provisoires
        df[colonne_ecart_type] = 0.0

        for moteur_id in df["unit_number"].unique():  # un moteur a la fois
            masque = df["unit_number"] == moteur_id  # lignes de ce moteur
            # rolling() est causal et ne porte que sur ce moteur
            df.loc[masque, colonne_moyenne] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).mean()
            df.loc[masque, colonne_ecart_type] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).std()

    return df


train = load_fd001("train")  # fichier d'entrainement FD001
sensor_cols = get_sensor_columns(train.columns)  # 21 capteurs bruts
constant_sensors = find_constant_sensors(train, sensor_cols)  # capteurs constants (Phase 1)
train = train.drop(columns=constant_sensors)  # on les retire
sensor_cols_restants = get_sensor_columns(train.columns)  # 15 capteurs informatifs

train, scaler = normalize_minmax(train, sensor_cols_restants)  # min-max (voir la limite notee en fin de notebook)
train = lisser_capteurs(train, sensor_cols_restants, fenetre=FENETRE)  # moyenne mobile par moteur
train = calculer_rul(train)  # RUL brute
train = plafonner_rul(train, plafond=PLAFOND_RUL)  # RUL plafonnee a 125

train = ajouter_features_glissantes(train, sensor_cols_restants, FENETRE)

# colonnes d'ecart-type et de moyenne glissantes, trouvees par boucle explicite
colonnes_ecart_type = []
colonnes_moyenne = []
for colonne in train.columns:
    if colonne.endswith("_glissante_std"):
        colonnes_ecart_type.append(colonne)
    if colonne.endswith("_glissante_moy"):
        colonnes_moyenne.append(colonne)

# au 1er cycle d'un moteur l'ecart-type n'existe pas : NaN remplace par 0 (comme en Phase 3)
train[colonnes_ecart_type] = train[colonnes_ecart_type].fillna(0.0)

feature_cols = sensor_cols_restants + colonnes_moyenne + colonnes_ecart_type + ["time_cycles"]
X = train[feature_cols]        # 46 features
y = train["RUL_cappee"]        # cible plafonnee a 125
groups = train["unit_number"]  # identifiant du moteur, sert uniquement a grouper

assert len(feature_cols) == 46, f"attendu 46 features, trouve {len(feature_cols)}"
assert X.isna().sum().sum() == 0, "il reste des NaN dans les features"
print(f"X : {X.shape}, y : {y.shape}, moteurs : {groups.nunique()}")

## 2. Outils de mesure

Quatre métriques, calculées sur le **même jeu** que celui qu'on évalue (train ou validation) :
- **MAE** : erreur absolue moyenne, en cycles ;
- **RMSE** : racine de l'erreur quadratique moyenne, en cycles (standard C-MAPSS) ;
- **R²** : part de la variance de la RUL expliquée (0 = aussi bon que prédire la moyenne) ;
- **score NASA moyen par prédiction** : le score NASA est une *somme* qui dépend du nombre de lignes ; on le divise par ce nombre pour pouvoir comparer des jeux de tailles différentes (protocole §1, point 5). C'est essentiel ici : en courbe d'apprentissage, le nombre de lignes d'entraînement change.

In [ ]:
def score_nasa(y_true, y_pred):
    '''
    Score NASA (Saxena et al., 2008), asymetrique : surestimer la RUL coute plus cher que la sous-estimer.
    (Copie conforme de la fonction du notebook 03.)

    Parameters
    ----------
    y_true : array-like
        Vraies RUL.
    y_pred : array-like
        RUL predites.

    Returns
    -------
    float
        Somme des penalites individuelles.
    '''
    total = 0.0  # on cumule la penalite de chaque exemple

    for vrai, predit in zip(y_true, y_pred):  # un exemple a la fois
        d = predit - vrai  # positif si surestimation
        if d < 0:  # sous-estimation : penalite douce
            total += np.exp(-d / 13) - 1
        else:  # surestimation : penalite severe
            total += np.exp(d / 10) - 1

    return total


def calculer_metriques(y_true, y_pred):
    '''
    Calcule les 4 metriques du diagnostic sur un jeu de prediction.

    Parameters
    ----------
    y_true : array-like
        Vraies RUL (plafonnees).
    y_pred : array-like
        RUL predites.

    Returns
    -------
    dict
        Cles "MAE", "RMSE", "R2", "NASA_moyen" (score NASA divise par le nombre de lignes).
    '''
    y_true = np.asarray(y_true, dtype=float)  # on travaille en tableaux numpy
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)  # erreur absolue moyenne
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))  # racine de l'erreur quadratique moyenne
    r2 = r2_score(y_true, y_pred)  # coefficient de determination
    nasa_moyen = score_nasa(y_true, y_pred) / len(y_true)  # score NASA par prediction

    return {"MAE": mae, "RMSE": rmse, "R2": r2, "NASA_moyen": nasa_moyen}


# sanity check : cas simple ou le calcul a la main est connu
metriques_test = calculer_metriques([100, 100], [120, 80])  # une surestimation et une sous-estimation de 20
assert abs(metriques_test["MAE"] - 20.0) < 1e-9
assert abs(metriques_test["RMSE"] - 20.0) < 1e-9
nasa_attendu = ((np.exp(2) - 1) + (np.exp(20 / 13) - 1)) / 2
assert abs(metriques_test["NASA_moyen"] - nasa_attendu) < 1e-6
print("metriques verifiees sur un cas a la main :", {k: round(float(v), 3) for k, v in metriques_test.items()})


noms_metriques = ["MAE", "RMSE", "R2", "NASA_moyen"]  # ordre des metriques dans les tableaux


def creer_modele():
    '''Cree un XGBoost avec les reglages de la Phase 3 (modele tout neuf a chaque appel).'''
    return XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)

## 3. Plis par moteur (fixés une fois)

Mêmes plis que partout (`GroupKFold(5, shuffle=True, random_state=42)`), calculés une fois. Vérification qu'aucun moteur n'est des deux côtés d'un pli.

In [ ]:
plis = list(GroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, y, groups=groups))

for i, (idx_train, idx_val) in enumerate(plis):  # un pli a la fois
    moteurs_train = set(groups.iloc[idx_train])
    moteurs_val = set(groups.iloc[idx_val])
    assert len(moteurs_train & moteurs_val) == 0, f"pli {i + 1} : un moteur est des deux cotes"
    assert len(moteurs_train) == 80 and len(moteurs_val) == 20, f"pli {i + 1} : tailles inattendues"
    print(f"Pli {i + 1} : {len(moteurs_train)} moteurs train, {len(moteurs_val)} moteurs validation, {len(idx_val)} lignes de validation")

## 4. Courbes d'apprentissage par nombre de moteurs d'entraînement

**Protocole** :
- pour chacun des 5 plis, les 20 moteurs de validation sont **tenus à part** et ne sont jamais tirés dans l'entraînement ; le « réservoir » d'entraînement est formé des 80 autres moteurs ;
- tailles testées : 10, 20, 40, 60, 80 moteurs ;
- pour chaque (pli, taille) < 80 : **5 tirages aléatoires** de moteurs dans le réservoir (graine déterministe) ; pour 80, un seul « tirage » (tout le réservoir) ;
- à chaque fois : modèle tout neuf, métriques sur le sous-ensemble d'entraînement (**train**) et sur les 20 moteurs de validation ;
- courbe : moyenne ± écart-type, d'abord par pli (moyenne des tirages d'un même pli), puis sur les 5 plis. L'écart-type est donc celui **entre plis**, comparable à celui du protocole §8.

**Attention à la lecture** : la validation porte toujours sur les mêmes 20 moteurs *pour un pli donné*, donc les tailles sont comparables **pli par pli** (test apparié possible).
Le point à 80 (tout le réservoir) doit redonner les chiffres de la Phase 3 (10.94 / 16.86) : c'est le contrôle de cohérence.

In [ ]:
TAILLES = [10, 20, 40, 60, 80]  # nombre de moteurs d'entrainement
NB_TIRAGES = 5                  # tirages aleatoires par (pli, taille) ; 1 seul pour la taille 80 (tout le reservoir)

resultats_lc = []  # une ligne par (pli, taille, tirage)

for i, (idx_train, idx_val) in enumerate(plis):  # un pli a la fois
    moteurs_reservoir = np.array(sorted(set(groups.iloc[idx_train])))  # les 80 moteurs disponibles pour l'entrainement
    X_val_pli = X.iloc[idx_val]
    y_val_pli = y.iloc[idx_val].values

    for j, taille in enumerate(TAILLES):  # une taille a la fois
        if taille == len(moteurs_reservoir):
            nb_tirages_ici = 1  # tout le reservoir : il n'y a rien a tirer
        else:
            nb_tirages_ici = NB_TIRAGES

        for tirage in range(nb_tirages_ici):  # un tirage a la fois
            generateur = np.random.RandomState(1000 * (i + 1) + 10 * j + tirage)  # graine deterministe par cas
            moteurs_tires = generateur.choice(moteurs_reservoir, size=taille, replace=False)  # sans remise

            masque_lignes = groups.iloc[idx_train].isin(moteurs_tires).values  # lignes des moteurs tires, dans le pli
            idx_sous_train = idx_train[masque_lignes]

            # garde-fous : seuls les moteurs tires sont utilises, jamais un moteur de validation
            moteurs_utilises = set(groups.iloc[idx_sous_train])
            assert len(moteurs_utilises) == taille, "mauvais nombre de moteurs d'entrainement"
            assert len(moteurs_utilises & set(groups.iloc[idx_val])) == 0, "un moteur de validation fuit dans l'entrainement"

            modele = creer_modele()  # modele tout neuf
            modele.fit(X.iloc[idx_sous_train], y.iloc[idx_sous_train])

            metriques_train = calculer_metriques(y.iloc[idx_sous_train].values, modele.predict(X.iloc[idx_sous_train]))
            metriques_val = calculer_metriques(y_val_pli, modele.predict(X_val_pli))

            ligne = {"pli": i + 1, "taille": taille, "tirage": tirage, "nb_lignes_train": len(idx_sous_train)}
            for nom_metrique in noms_metriques:
                ligne["train_" + nom_metrique] = metriques_train[nom_metrique]
                ligne["val_" + nom_metrique] = metriques_val[nom_metrique]
            resultats_lc.append(ligne)

    print(f"Pli {i + 1} termine")

lc = pd.DataFrame(resultats_lc)
print(f"{len(lc)} entrainements au total")
assert len(lc) == 5 * (4 * NB_TIRAGES + 1), "nombre d'entrainements inattendu"

# sanity check : a 80 moteurs (tout le reservoir), on doit retrouver les chiffres de la Phase 3
lc_80 = lc[lc["taille"] == 80]
assert abs(lc_80["val_RMSE"].mean() - 16.86) < 0.05, f"RMSE val moyen a 80 moteurs {lc_80['val_RMSE'].mean():.2f} != 16.86"
assert abs(lc_80["train_RMSE"].mean() - 10.94) < 0.05, f"RMSE train moyen a 80 moteurs {lc_80['train_RMSE'].mean():.2f} != 10.94"
print("taille 80 = chiffres de la Phase 3 (10.94 / 16.86) : OK")

In [ ]:
# agregation : (1) moyenne des tirages dans chaque pli, (2) moyenne et ecart-type ENTRE plis
par_pli_et_taille = lc.groupby(["taille", "pli"]).mean(numeric_only=True).reset_index()  # moyenne des tirages d'un pli

tableau_lc = []
for taille in TAILLES:
    sous_tableau = par_pli_et_taille[par_pli_et_taille["taille"] == taille]
    ligne = {"nb_moteurs": taille}
    for nom_metrique in noms_metriques:
        ligne["train_" + nom_metrique + "_moy"] = sous_tableau["train_" + nom_metrique].mean()
        ligne["train_" + nom_metrique + "_std"] = sous_tableau["train_" + nom_metrique].std(ddof=0)
        ligne["val_" + nom_metrique + "_moy"] = sous_tableau["val_" + nom_metrique].mean()
        ligne["val_" + nom_metrique + "_std"] = sous_tableau["val_" + nom_metrique].std(ddof=0)
    # variabilite due au tirage des moteurs, a pli fixe (0 pour la taille 80, un seul tirage)
    ecarts_tirage = []
    for pli_numero in range(1, 6):
        valeurs_pli = lc[(lc["taille"] == taille) & (lc["pli"] == pli_numero)]["val_RMSE"]
        ecarts_tirage.append(valeurs_pli.std(ddof=0))
    ligne["val_RMSE_std_tirages"] = np.mean(ecarts_tirage)
    tableau_lc.append(ligne)
tableau_lc = pd.DataFrame(tableau_lc)

# affichage lisible : RMSE, MAE, R2, NASA moyen en "moyenne +/- ecart-type entre plis"
print(f"{'moteurs':>7s} | {'RMSE train':>14s} | {'RMSE val':>14s} | {'MAE val':>14s} | {'R2 val':>14s} | {'NASA moy val':>16s} | std tirages (RMSE val)")
for i in range(len(tableau_lc)):
    r = tableau_lc.iloc[i]
    print(f"{int(r['nb_moteurs']):7d} | "
          f"{r['train_RMSE_moy']:6.2f} +/- {r['train_RMSE_std']:4.2f} | "
          f"{r['val_RMSE_moy']:6.2f} +/- {r['val_RMSE_std']:4.2f} | "
          f"{r['val_MAE_moy']:6.2f} +/- {r['val_MAE_std']:4.2f} | "
          f"{r['val_R2_moy']:6.3f} +/- {r['val_R2_std']:5.3f} | "
          f"{r['val_NASA_moyen_moy']:7.2f} +/- {r['val_NASA_moyen_std']:5.2f} | "
          f"{r['val_RMSE_std_tirages']:.2f}")

In [ ]:
# reference d'echelle : RMSE du modele "bete" qui predit la moyenne du train (meme calcul que notebook 03 section 13)
rmse_baseline_plis = []
for idx_train, idx_val in plis:
    moyenne_train = y.iloc[idx_train].mean()  # moyenne du train du pli, jamais celle de la validation
    rmse_baseline_plis.append(np.sqrt(np.mean((y.iloc[idx_val].values - moyenne_train) ** 2)))
rmse_baseline = np.mean(rmse_baseline_plis)
print(f"RMSE de la baseline moyenne : {rmse_baseline:.2f}")

# figure : courbe d'apprentissage RMSE, moyenne +/- ecart-type entre plis
tailles_x = np.array(tableau_lc["nb_moteurs"])
train_moy = np.array(tableau_lc["train_RMSE_moy"])
train_std = np.array(tableau_lc["train_RMSE_std"])
val_moy = np.array(tableau_lc["val_RMSE_moy"])
val_std = np.array(tableau_lc["val_RMSE_std"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tailles_x, train_moy, marker="o", color="tab:blue", label="RMSE train (moteurs appris)")
ax.fill_between(tailles_x, train_moy - train_std, train_moy + train_std, color="tab:blue", alpha=0.2)
ax.plot(tailles_x, val_moy, marker="o", color="tab:orange", label="RMSE validation (moteurs jamais vus)")
ax.fill_between(tailles_x, val_moy - val_std, val_moy + val_std, color="tab:orange", alpha=0.2)
ax.axhline(rmse_baseline, linestyle=":", color="grey", label=f"baseline « prédire la moyenne » ({rmse_baseline:.1f})")

ax.set_xlabel("nombre de moteurs d'entraînement")
ax.set_ylabel("RMSE (cycles), moyenne ± écart-type entre 5 plis")
ax.set_xticks(TAILLES)
ax.set_ylim(0, rmse_baseline + 5)
ax.set_title("Courbe d'apprentissage XGBoost (46 features, FD001)")
ax.legend(loc="center right")

plt.tight_layout()
plt.savefig("../docs/figures/04c_courbe_apprentissage_rmse.png", dpi=120)
plt.show()

## 5. Quantifier : la courbe baisse-t-elle encore ?

On compare le **gain de RMSE de validation entre tailles successives** au **bruit entre plis** (écart-type).
Pour le dernier palier (60 → 80), on regarde aussi si le gain va dans le même sens sur les 5 plis : un pli garde les mêmes 20 moteurs de validation à toutes les tailles, donc la comparaison pli par pli est légitime.

Pas de test statistique ni d'extrapolation à 100 / 200 moteurs : avec 5 plis et 5 points de courbe, ils seraient trop fragiles pour peser dans la décision.

In [ ]:
# 1) gains entre tailles successives
print("Gain de RMSE validation entre tailles successives (positif = amelioration), a comparer au bruit entre plis :")
for k in range(1, len(TAILLES)):
    gain = val_moy[k - 1] - val_moy[k]
    print(f"  {TAILLES[k - 1]:2d} -> {TAILLES[k]:2d} moteurs : {gain:+.2f}   (ecart-type entre plis a {TAILLES[k]} : {val_std[k]:.2f})")

# 2) dernier palier, pli par pli : le gain va-t-il dans le meme sens partout ?
def rmse_val_par_pli(taille):
    '''RMSE de validation de chaque pli (moyenne des tirages) pour une taille donnee, tries par pli.'''
    sous = par_pli_et_taille[par_pli_et_taille["taille"] == taille].sort_values("pli")
    return sous["val_RMSE"].values

gains_par_pli = rmse_val_par_pli(60) - rmse_val_par_pli(80)  # positif = 80 moteurs font mieux que 60 sur ce pli
nb_plis_en_gain = 0
for gain in gains_par_pli:  # un pli a la fois
    if gain > 0:
        nb_plis_en_gain += 1
print(f"\n60 -> 80 moteurs, gain par pli : {np.round(gains_par_pli, 2)} ({nb_plis_en_gain}/5 plis en amelioration)")

## 6. Conclusion chiffrée

**Ce que montre la courbe** (figure `docs/figures/04c_courbe_apprentissage_rmse.png`, tableau de la section 4) :

| Moteurs d'entraînement | RMSE train | RMSE validation (± écart-type entre plis) | Écart train/val | Gain de validation vs taille précédente |
|---|---|---|---|---|
| 10 | 1.99 | 21.16 (±2.17) | 19.17 | — |
| 20 | 4.74 | 19.71 (±2.38) | 14.97 | +1.45 |
| 40 | 8.27 | 17.86 (±1.75) | 9.59 | +1.85 |
| 60 | 9.87 | 17.19 (±1.29) | 7.32 | +0.67 |
| 80 | 10.94 | 16.86 (±1.25) | 5.92 | +0.33 |

1. **L'erreur de validation baisse encore à 80 moteurs, mais de moins en moins** : −1.85 de 20 à 40 moteurs, −0.67 de 40 à 60, −0.33 de 60 à 80, soit environ 0.017 cycle de RMSE par moteur ajouté sur le dernier palier. Les autres métriques de validation vont dans le même sens (MAE 14.56 → 11.49, R² 0.738 → 0.835).
2. **Ce dernier gain de 0.33 est petit devant le bruit** (écart-type entre plis 1.25), **mais il est de même signe sur les 5 plis** (de +0.18 à +0.45). C'est une indication que la courbe n'est pas tout à fait plate, pas une mesure de ce qu'apporteraient 20 ou 100 moteurs de plus.
3. **L'écart train/validation se réduit** (19.2 → 5.9) parce que le train *monte* (le modèle ne peut plus mémoriser autant de moteurs) et que la validation *descend*. Il reste de **5.9 cycles à 80 moteurs** : un sur-apprentissage résiduel subsiste, qu'une régularisation ou des arbres moins profonds pourraient réduire (piste déjà notée en Phase 3). L'erreur train ne tombe pas à 0 pour autant (RMSE 10.94 sur des moteurs déjà vus) : une part de l'erreur vient d'information absente des features (bruit capteur, variations de fabrication), pas du sur-apprentissage.
**Réponse à « ajouter FD003 aiderait-il ? » : probablement un peu, pas beaucoup.** Le modèle est déjà à 17.86 avec 40 moteurs (la baseline est à 41.69) et la baisse restante se compte en dixièmes de cycle par vingt moteurs, sous le bruit entre plis. **Le plancher d'environ 16.5 observé en Phase 3 vient donc surtout des features et du modèle, pas du volume de données** : c'est là qu'il faut travailler en priorité (features, modèle de séquence, hyperparamètres) plutôt que de chercher plus de moteurs du même type.

**Mais FD003 n'est pas « plus de FD001 »** : d'après `data/readme.txt`, FD003 a la même condition de vol (une seule) mais **deux modes de panne** (dégradation du compresseur haute pression *et* du ventilateur) contre un seul pour FD001. Cette courbe mesure l'effet de plus de moteurs *du même type*. Elle ne dit rien de l'effet de moteurs à autre loi de dégradation sur l'erreur en FD001 (qui pourrait aider, être neutre ou gêner). Cette question demande une expérience dédiée (entraîner sur FD001 + FD003, valider sur FD001), que ce notebook ne fait pas.

**Pour un futur LSTM / CNN 1D** : cette courbe ne prouve pas que 100 moteurs suffisent. Elle montre seulement qu'un modèle à arbres atteint presque son plancher dès 40-60 moteurs. Un réseau a beaucoup plus de paramètres et est généralement plus gourmand en données : sa courbe peut être bien plus pentue. La procédure de la section 4 est réutilisable telle quelle (remplacer `creer_modele()`) ; refaire cette courbe avec le réseau est la bonne façon de répondre. Si un LSTM plafonne comme XGBoost, cela pointerait vers un problème d'information (features, bruit) plus que de quantité de données. Chaque moteur étant une séquence, la taille effective du jeu d'entraînement se compte en moteurs (100), pas en lignes (20 631), car les cycles d'un même moteur sont fortement corrélés.

## 7. Limites et problèmes de méthode relevés (règle 11 d'`AGENTS.md`)

1. **Taille maximale 80, pas 100** : les 20 moteurs de validation sont retirés du réservoir. Ce qui se passerait à 100 ou 200 moteurs n'est pas mesuré ici.
2. **Peu de plis** : 5 plis donnent un écart-type fragile. Un résultat plus robuste répéterait la validation avec plusieurs graines de plis (protocole §8), non fait ici. Le bruit dû au tirage des moteurs (écart-type à pli fixe : 1.56 à 10 moteurs, 1.18 à 20, 0.53 à 40, 0.45 à 60) est du même ordre que le gain 60 → 80 (0.33) ; 5 tirages par taille réduisent ce bruit sans l'éliminer.
3. **Hyperparamètres fixés** (200 arbres, profondeur 5) à toutes les tailles. La complexité optimale dépend du nombre de données : les petites tailles préfèrent sans doute un modèle plus simple, les grandes un plus complexe. La courbe est donc pessimiste à petite taille, et ne dit pas ce qu'un modèle *réglé* ferait avec plus de données. À croiser avec la session d'optimisation des hyperparamètres.
4. **Validation sur tous les cycles, cible plafonnée à 125** : ce n'est pas la mesure du test officiel (un point par moteur, dernier cycle, RUL non plafonnée). Les conclusions sur le volume de données ne se transfèrent pas forcément à l'identique. Le jeu de test officiel n'est pas utilisé ici.
5. **Normalisation ajustée sur les 100 moteurs avant la validation croisée** (`normalize_minmax` appelé une fois sur tout le train, hérité de la Phase 3), alors que le protocole §1, point 3, demande de refaire le `fit` dans chaque pli. **Sans effet pour XGBoost** (un arbre ne dépend que de l'ordre des valeurs, que min-max conserve), mais **à corriger avant tout Ridge, LSTM ou CNN 1D**, sinon les résultats seraient légèrement optimistes. Ce notebook ne touche pas à `src/` : à traiter ailleurs.
6. **Une seule famille de features** (les 46 de la Phase 3) : la courbe ne dit pas si d'autres features abaisseraient le plancher (objet de la session features).
7. **Un seul run** par configuration. Les chiffres à 80 moteurs reproduisent ceux de la Phase 3 (10.94 / 16.86, vérifiés par `assert`), mais XGBoost peut varier légèrement d'une machine à l'autre (note de reproductibilité de `docs/03_modelisation.md`).
8. **Points du protocole non couverts ici** (hors périmètre de cette tâche, listés pour mémoire) : baselines du §2 autres que la moyenne, métriques de classification (§4), autres métriques de régression du §3 (médiane, max, biais, tranches de RUL), test de significativité entre modèles (§8).